In [0]:
from pyspark.sql.functions import col, when, count, trim, initcap, to_date, to_timestamp

# 1. Cargar datos 
df = spark.table("default.ingesta_datosV3")

# 2. Diagnóstico de nulos por columna 
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

# 3. Tipado correcto de columnas 
df = (
    df
    .withColumn("fecha_cit", to_date(col("fecha_cit"), "yyyy-MM-dd"))
    .withColumn("fechaNac_pac", to_date(col("fechaNac_pac"), "yyyy-MM-dd"))
    .withColumn("hr_inicio_cit", col("hr_inicio_cit").cast("string"))  
    .withColumn("hr_fin_cit", col("hr_fin_cit").cast("string"))
    .withColumn("mins_cit", col("mins_cit").cast("integer"))
    .withColumn("edad_pac_cita", col("edad_pac_cita").cast("integer"))
    .withColumn("peso_kg", col("peso_kg").cast("double"))
    .withColumn("altura_m", col("altura_m").cast("double"))
    .withColumn("imc", col("imc").cast("double"))
    .withColumn("pago_clie", col("pago_clie").cast("double"))
    .withColumn("pago_aseg", col("pago_aseg").cast("double"))
    .withColumn("pago_total", col("pago_total").cast("double"))
)

# --- 4. Manejo de nulos específico (no fillna global) ---
# nom_aseguradora nula = paciente sin seguro / pago particular
df = df.fillna({"nom_aseguradora": "Particular / Sin Seguro"})

print("Limpieza de texto")
# --- 5. Limpieza de texto (espacios extra, capitalización consistente) ---
df = (
    df
    .withColumn("nom_comp_pac", trim(initcap(col("nom_comp_pac"))))
    .withColumn("nom_comp_doc", trim(initcap(col("nom_comp_doc"))))
    .withColumn("motivo_cita", trim(col("motivo_cita")))
    .withColumn("especialidad_medica", trim(col("especialidad_medica")))
    .withColumn("nom_sucursal", trim(col("nom_sucursal")))
)

#6. Eliminar duplicados 
df = df.dropDuplicates()

# 7. Validación de anomalías (filas que no deberían existir) 
# Pagos negativos
df.filter((col("pago_clie") < 0) | (col("pago_aseg") < 0) | (col("pago_total") < 0)).display()

print("Consistencia: pago_total debe ser igual a pago_clie + pago_aseg")
# Consistencia: pago_total debe ser igual a pago_clie + pago_aseg
df.filter(col("pago_total") != (col("pago_clie") + col("pago_aseg"))).display()

print("Edad fuera de rango razonable")
# Edad fuera de rango razonable (0-110 años)
df.filter((col("edad_pac_cita") < 0) | (col("edad_pac_cita") > 110)).display()

print("Duración de cita fuera de valores esperados")
# Duración de cita fuera de valores esperados (15/30/45/60 min)
df.filter(~col("mins_cit").isin(15, 30, 45, 60)).display()

print("Verificación final de nulos")
# 8. Verificación final de nulos (debe quedar en 0 en todas las columnas) 
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

# 9. Guardar tabla limpia en el catálogo para el resto del equipo 
df.write.mode("overwrite").saveAsTable("workspace.default.citas_pmm_limpioV3")

print(f"Filas finales: {df.count()}")


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2493,0,0,0


Limpieza de texto


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


Consistencia: pago_total debe ser igual a pago_clie + pago_aseg


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total
2025-05-09,2026-07-15 16:45:00,2026-07-15 17:00:00,15,Sinusitis Crónica,Completada,Pedro Vidal Torres,M,2024-04-08,1,7.3,0.62,19.0,O+,Dr. Silvia Santana Vargas,Otorrinolaringología,PMM Costa del Este,PALIG,19.29,53.2,72.49
2025-04-18,2026-07-15 08:30:00,2026-07-15 08:45:00,15,Control de Función Renal,Completada,Gabriel Gomez Torres,M,1964-08-27,60,69.8,1.63,26.3,A+,Dr. Roberto Rios Cabrera,Nefrología,PMM Brisas del Golf,PALIG,9.98,43.25,53.23
2025-06-09,2026-07-15 09:45:00,2026-07-15 10:45:00,60,Manejo de Ansiedad,Completada,Ana Mendez Perez,F,1961-04-05,64,67.3,1.68,23.8,A+,Dr. Javier Hernandez Rodriguez,Psiquiatría,PMM Costa del Este,MAPFRE Panamá,42.18,57.83,100.01
2025-08-21,2026-07-15 16:15:00,2026-07-15 16:45:00,30,Limpieza Dental,Completada,Luis Rodriguez Suarez,M,1992-09-22,32,75.4,1.74,24.9,O+,Dr. Juan Flores Castro,Odontología,PMM Tocumen,Seguros ASSA,9.81,18.75,28.56
2025-12-13,2026-07-15 16:30:00,2026-07-15 16:45:00,15,Mamografía,Completada,Marta Martinez Garcia,F,1988-04-08,37,74.9,1.63,28.2,B+,Dr. Eduardo Iglesias Lopez,Radiología,PMM San Francisco,Compañía Internacional de Seguros (IS),24.33,61.13,85.46
2025-04-22,2026-07-15 08:00:00,2026-07-15 08:15:00,15,Dolor de Pecho,Completada,Javier Martinez Cruz,M,1962-05-29,62,43.2,1.75,14.1,O+,Dr. Lucia Sanz Rojas,Cardiología,PMM El Dorado,PALIG,11.64,61.63,73.27
2025-09-30,2026-07-15 17:30:00,2026-07-15 17:45:00,15,Insuficiencia Renal,Completada,Manuel Gonzalez Ortiz,M,1996-05-23,29,55.4,1.55,23.1,A+,Dr. Daniel Morales Soto,Nefrología,PMM Costa del Este,Seguros SURA,40.37,53.58,93.95
2025-09-30,2026-07-15 09:15:00,2026-07-15 10:15:00,60,Cálculos Renales,Completada,Mario Vidal Santana,M,1976-02-22,49,101.0,1.79,31.5,A+,Dr. Carmen Torres Romero,Nefrología,PMM Costa del Este,Seguros ASSA,42.15,120.21,162.36
2025-11-27,2026-07-15 08:45:00,2026-07-15 09:45:00,60,Lesión,Completada,Laura Rubio Cabrera,F,1972-04-03,53,68.9,1.59,27.3,A+,Dr. Ana Ramos Suarez,Fisioterapia,PMM San Francisco,Seguros SURA,41.88,49.67,91.55
2025-03-07,2026-07-15 08:15:00,2026-07-15 09:00:00,45,Ultrasonido Abdominal,Completada,Pedro Navarro Ruiz,M,1995-09-18,29,79.8,1.71,27.3,O+,Dr. Alejandro Mendez Aguilar,Radiología,PMM El Dorado,Seguros SURA,13.9,49.44,63.34


...

Duración de cita fuera de valores esperados


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


Verificación final de nulos


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Filas finales: 10000
